In [ ]:
import torch
import yaml
from sfm.data.moving_librimix import MovingLibriMix as DS
from sfm.data.base_datamodule import BaseDatamodule as DM
import matplotlib as mpl
from matplotlib.animation import PillowWriter
import matplotlib.pyplot as plt
from matplotlib.path import Path
import numpy as np
from IPython.display import HTML
from itertools import islice
from sfm.data.utils import animate_trajectory
import os


LENGTH = 30.0
EXAMPLE_IDX = 5
GIF_PATH = '../animations/'
CONFIG_PATH = '../../config/'

In [ ]:
# subclass for trajectories
class GIFTrajectory(DS):
    def __init__(
        self,
        audio_time: float = LENGTH,  # in sec
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.audio_time = audio_time
    
    def __getitem__(self, idx):
        # get meta
        example = self.init_example(idx)

        example['audio_time'] = self.audio_time
        example['audio_samples'] = int(self.audio_time * self.sample_rate)

        # get room simulation setup
        self.get_scenario(example)

        return example
    
# instantiate dataset
config_path = CONFIG_PATH + 'base_config.yaml'
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)
ds = GIFTrajectory(dataset_name='test', **config)

In [ ]:
# wrapper to export GIF
def export_trajectory_gif(
    example,
    gif_name: str,
    n_speaker: int = 2,
    wall_dist: float = 0.5,
    fps: int = 20,
    dpi: int = 100,
    loop: int = 0,
    **kwargs,
):
    ani = animate_trajectory(
        example,
        n_speaker=n_speaker,
        wall_dist=wall_dist,
        fps=fps,
        dpi=dpi,
        **kwargs,
    )

    output_path = GIF_PATH + f'{gif_name}.gif'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    writer = PillowWriter(fps=fps)
    ani.save(output_path, writer=writer, dpi=dpi)

In [ ]:
# show in notebook
display(HTML(animate_trajectory(
    ds[EXAMPLE_IDX], 
    n_speaker=2, 
    wall_dist=config['data_params']['acoustic_params']['speaker']['wall_dist'],
    dpi=30,
    show_path=True, 
    show_forces=True, 
    show_boundaries=True, 
    show_driving_rect=True,
).to_html5_video()))

In [ ]:
# export as GIF
export_trajectory_gif(
    ds[EXAMPLE_IDX],
    f'example_{EXAMPLE_IDX}',
    n_speaker=2,
    dpi=30,
    show_path=True, 
    show_forces=True, 
    show_boundaries=True, 
    show_driving_rect=True,
)